In [4]:
import numpy as np
import time
from multiprocessing import Pool, cpu_count

In [5]:
# Suma un chunk de los arreglos
def suma_chunk(args):
    A_chunk, B_chunk, start_idx = args
    result = []
    for i in range(len(A_chunk)):
        result.append((start_idx + i, A_chunk[i] + B_chunk[i]))
    return result

In [6]:
# Suma paralela usando multiprocessing
def suma_paralela_multiprocessing(A, B, num_processes=None):
    if num_processes is None:
        num_processes = cpu_count()

    N = len(A)
    chunk_size = N // num_processes

    # Dividir arreglos en chunks
    chunks = []
    for i in range(num_processes):
        start = i * chunk_size
        end = start + chunk_size if i < num_processes - 1 else N
        chunks.append((A[start:end], B[start:end], start))

    # Procesar en paralelo
    with Pool(processes=num_processes) as pool:
        results = pool.map(suma_chunk, chunks)

    # Combinar resultados
    R = np.zeros(N, dtype=int)
    for chunk_result in results:
        for idx, valor in chunk_result:
            R[idx] = valor

    return R

In [8]:
# Configuración
N = 10000
num_cores = cpu_count()

print(f"Núcleos disponibles: {num_cores}")
print(f"Tamaño de arreglos: {N} elementos\n")

Núcleos disponibles: 2
Tamaño de arreglos: 10000 elementos



In [10]:
# Crear arreglos
A = np.random.randint(0, 101, size=N)
B = np.random.randint(0, 101, size=N)

In [11]:
# Mostrar primeros elementos
print("Primeros 10 elementos:")
print("=" * 50)
print(f"{'Índice':<10} {'A':<10} {'B':<10}")
print("=" * 50)
for i in range(10):
    print(f"{i:<10} {A[i]:<10} {B[i]:<10}")

Primeros 10 elementos:
Índice     A          B         
0          78         58        
1          76         42        
2          58         23        
3          10         44        
4          95         32        
5          39         97        
6          67         63        
7          96         85        
8          80         70        
9          12         66        


In [12]:
# Suma secuencial
print("\n Ejecutando suma secuencial...")
inicio_sec = time.time()
R_sec = A + B
fin_sec = time.time()
tiempo_secuencial = fin_sec - inicio_sec


 Ejecutando suma secuencial...


In [13]:
# Suma paralela
print(" Ejecutando suma paralela (multiprocessing)...")
inicio_par = time.time()
R_par = suma_paralela_multiprocessing(A, B, num_cores)
fin_par = time.time()
tiempo_paralelo = fin_par - inicio_par

 Ejecutando suma paralela (multiprocessing)...


In [14]:
# Resultados
print("\n" + "=" * 60)
print("RESULTADOS (Primeros 10 elementos):")
print("=" * 60)
print(f"{'Índice':<10} {'A':<10} {'B':<10} {'R (A+B)':<10}")
print("=" * 60)
for i in range(10):
    print(f"{i:<10} {A[i]:<10} {B[i]:<10} {R_par[i]:<10}")


RESULTADOS (Primeros 10 elementos):
Índice     A          B          R (A+B)   
0          78         58         136       
1          76         42         118       
2          58         23         81        
3          10         44         54        
4          95         32         127       
5          39         97         136       
6          67         63         130       
7          96         85         181       
8          80         70         150       
9          12         66         78        


In [19]:
# Verificación
if np.array_equal(R_sec, R_par):
    print("\n Verificación exitosa")
else:
    print("\n Error en resultados")


 Verificación exitosa


In [18]:
# Estadísticas
print("\n" + "=" * 60)
print("Resultados")
print("=" * 60)
print(f"Elementos procesados: {N}")
print(f"Núcleos utilizados: {num_cores}")
print(f"Tiempo secuencial: {tiempo_secuencial:.6f} segundos")
print(f"Tiempo paralelo: {tiempo_paralelo:.6f} segundos")
print(f"Speedup: {tiempo_secuencial/tiempo_paralelo:.2f}x")
print("=" * 60)


Resultados
Elementos procesados: 10000
Núcleos utilizados: 2
Tiempo secuencial: 0.000232 segundos
Tiempo paralelo: 0.063978 segundos
Speedup: 0.00x
